# ABS Business Entries and Exits 8165.0

Quarterly counts of business entries, exits and total operating businesses on the ATO ABN/GST register. Source: ABS 8165.0 time-series workbook TS13 (last published in the Jul-2020 to Jun-2024 release). Coverage: 2013Q1 – 2025Q1.


## Python setup

In [1]:
# third-party
import pandas as pd
import readabs as ra
import mgplot as mg

pd.options.display.max_rows = 9999
pd.options.display.max_columns = 999

In [2]:
SHOW = False
CHART_DIR = "./CHARTS/8165.0 - Counts of Australian Businesses/"
SOURCE = "ABS: 8165.0 (TS13)"
FOOTER = (
    "Australia. Counts of actively trading businesses on the ATO ABN/GST register. "
)

mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()

## Fetch data

The current release (2021–2025) dropped the time-series workbook. The previous release (2020–2024) is the last to publish TS13 with quarterly data back to 2013.

In [3]:
RELEASE = (
    "https://www.abs.gov.au/statistics/economy/business-indicators/"
    "counts-australian-businesses-including-entries-and-exits/jul2020-jun2024"
)
raw = ra.grab_abs_url(url=RELEASE, single_excel_only="8165TS13", get_zip=False)
sheet = raw["8165TS13---Data1"]

# Build a clean quarterly DataFrame with PeriodIndex
def parse_ts(sheet: pd.DataFrame) -> pd.DataFrame:
    # Row 8 of the sheet has the Series IDs; data starts at row 9
    series_ids = sheet.iloc[8, 1:].tolist()
    body = sheet.iloc[9:].copy()
    body.index = pd.to_datetime(body.iloc[:, 0])
    body.index = pd.PeriodIndex(body.index, freq="Q-DEC")
    body = body.iloc[:, 1:].apply(pd.to_numeric, errors="coerce")
    body.columns = [
        "exits_orig", "entries_orig", "total_orig",
        "exits_sa",   "entries_sa",   "total_sa",
    ]
    return body

ts = parse_ts(sheet)
print(ts.shape)
ts.tail(6)

(49, 6)


,exits_orig,entries_orig,total_orig,exits_sa,entries_sa,total_sa
Unnamed: 0,,,,,,
2023Q4,137846,131280,2611100,104873,132434,2624820
2024Q1,84351,110368,2637117,100211,123591,2646118
2024Q2,104952,130046,2662211,127410,130716,2657688
2024Q3,112019,142199,2692391,109682,126861,2673812
2024Q4,142073,122844,2674085,108108,123788,2688638
2025Q1,94806,115068,2694347,112827,129273,2702938


## Compute entry and exit rates

Quarterly rate = flow during the quarter / opening stock (= close of previous quarter).

In [4]:
def make_rates(ts: pd.DataFrame, suffix: str = "sa") -> pd.DataFrame:
    entries = ts[f"entries_{suffix}"]
    exits = ts[f"exits_{suffix}"]
    total = ts[f"total_{suffix}"]
    opening = total.shift(1)
    return pd.DataFrame({
        "Entry rate":   entries / opening * 100,
        "Exit rate":    exits   / opening * 100,
    }).dropna()


def annualise(quarterly_rates: pd.DataFrame) -> pd.DataFrame:
    """Convert quarterly rates to a trailing 4-quarter sum."""
    return quarterly_rates.rolling(4).sum().dropna()


q_rates = make_rates(ts, suffix="sa")
a_rates = annualise(q_rates)
print("latest annualised rates:")
print(a_rates.tail(4).round(2))

latest annualised rates:
            Entry rate  Exit rate
Unnamed: 0                       
2024Q2           19.64      17.00
2024Q3           19.52      16.79
2024Q4           19.05      16.80
2025Q1           19.15      17.18


## Plot

In [5]:
def plot_annualised_rates() -> None:
    mg.line_plot_finalise(
        a_rates,
        title="Australia: business entry and exit rates (annualised)",
        ylabel="Per cent of opening stock\n(4-quarter trailing sum)",
        rfooter=SOURCE,
        lfooter=FOOTER + "Seasonally adjusted. ",
        width=2,
        annotate=True,
        rounding=1,
        legend={"loc": "best", "fontsize": "small"},
        show=SHOW,
    )


def plot_quarterly_rates() -> None:
    mg.line_plot_finalise(
        q_rates,
        title="Australia: business entry and exit rates, quarterly",
        ylabel="Per cent of opening stock",
        rfooter=SOURCE,
        lfooter=FOOTER + "Seasonally adjusted. ",
        width=1.5,
        annotate=True,
        rounding=2,
        legend={"loc": "best", "fontsize": "small"},
        show=SHOW,
    )


def plot_net_churn() -> None:
    net = (q_rates["Entry rate"] - q_rates["Exit rate"]).rename("Net entry rate (entries − exits)")
    mg.line_plot_finalise(
        net,
        title="Australia: net business entry rate (entries minus exits)",
        ylabel="Per cent of opening stock, quarterly",
        rfooter=SOURCE,
        lfooter=FOOTER + "Seasonally adjusted. ",
        width=2,
        annotate=True,
        rounding=2,
        y0=True,
        show=SHOW,
    )


def plot_counts() -> None:
    counts = ts[["entries_sa", "exits_sa"]].rename(columns={
        "entries_sa": "Entries", "exits_sa": "Exits",
    }) / 1_000.0
    mg.line_plot_finalise(
        counts,
        title="Australia: quarterly business entries and exits, counts",
        ylabel="Thousands of businesses per quarter",
        rfooter=SOURCE,
        lfooter=FOOTER + "Seasonally adjusted. ",
        width=1.5,
        annotate=True,
        rounding=0,
        legend={"loc": "best", "fontsize": "small"},
        show=SHOW,
    )


plot_annualised_rates()
plot_quarterly_rates()
plot_net_churn()
plot_counts()

## Finished

In [6]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-05-21 11:15:36

Python implementation: CPython
Python version       : 3.14.2
IPython version      : 9.13.0

conda environment: n/a

Compiler    : Clang 21.1.4 
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot : 0.2.23
pandas : 3.0.2
readabs: 0.1.8

Watermark: 2.6.0

